# XGBoost


##  XGBoost - Regression

* Building an XGBoost classifier using `numpy` only. Training the XGBoost model on the `California Housing` regression task. Reporting on the performance predicting unseen test samples.

In [ ]:
import numpy as np


class TreeNode:
    """Represents a node in the decision tree"""
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.left = None
        self.right = None
        self.value = None  # Leaf value
        self.is_leaf = False


class DecisionTree:
    """Decision tree for XGBoost"""

    def __init__(self, max_depth=3, min_samples_leaf=20, lambda_=1.0, gamma=0.0):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.lambda_ = lambda_
        self.gamma = gamma
        self.root = None

    def _calc_leaf_value(self, gradients, hessians):
        G = gradients.sum()
        H = hessians.sum()
        return - G / (H + self.lambda_)

    def _calc_gain(self, G_left, H_left, G_right, H_right, G_total, H_total):
        # XGBoost split gain (second-order Taylor)
        lambda_ = self.lambda_
        gamma = self.gamma
        left_score = G_left**2 / (H_left + lambda_)
        right_score = G_right**2 / (H_right + lambda_)
        parent_score = G_total**2 / (H_total + lambda_)
        gain = 0.5 * (left_score + right_score - parent_score) - gamma
        return gain

    def _build_tree(self, X, gradients, hessians, depth):
        node = TreeNode()

        G_total = gradients.sum()
        H_total = hessians.sum()

        # Stopping criteria: max depth or too few samples
        if depth >= self.max_depth or X.shape[0] <= self.min_samples_leaf:
            node.is_leaf = True
            node.value = self._calc_leaf_value(gradients, hessians)
            return node

        n_samples, n_features = X.shape
        best_gain = 0.0
        best_feature = None
        best_threshold = None

        # Search best split over all features
        for j in range(n_features):
            xj = X[:, j]
            # Sort by feature j
            order = np.argsort(xj)
            xj_sorted = xj[order]
            g_sorted = gradients[order]
            h_sorted = hessians[order]

            # Prefix sums for left side
            G_left_cum = np.cumsum(g_sorted)
            H_left_cum = np.cumsum(h_sorted)

            # Total G/H
            G_total_j = G_total
            H_total_j = H_total

            # Try splits between unique values, avoid splits too close to ends
            for i in range(self.min_samples_leaf,
                           n_samples - self.min_samples_leaf):
                if xj_sorted[i] == xj_sorted[i - 1]:
                    continue  # skip identical thresholds

                G_left = G_left_cum[i - 1]
                H_left = H_left_cum[i - 1]
                G_right = G_total_j - G_left
                H_right = H_total_j - H_left

                gain = self._calc_gain(G_left, H_left, G_right, H_right,
                                       G_total_j, H_total_j)
                if gain > best_gain:
                    best_gain = gain
                    best_feature = j
                    best_threshold = (xj_sorted[i] + xj_sorted[i - 1]) / 2.0

        # If no positive gain, make leaf
        if best_feature is None:
            node.is_leaf = True
            node.value = self._calc_leaf_value(gradients, hessians)
            return node

        # Split data according to best feature / threshold
        mask_left = X[:, best_feature] <= best_threshold
        X_left, X_right = X[mask_left], X[~mask_left]
        g_left, g_right = gradients[mask_left], gradients[~mask_left]
        h_left, h_right = hessians[mask_left], hessians[~mask_left]

        node.feature_index = best_feature
        node.threshold = best_threshold
        node.left = self._build_tree(X_left, g_left, h_left, depth + 1)
        node.right = self._build_tree(X_right, g_right, h_right, depth + 1)
        return node

    def fit(self, X, gradients, hessians):
        """Build the tree"""
        self.root = self._build_tree(X, gradients, hessians, depth=0)

    def _predict_row(self, x, node):
        if node.is_leaf:
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._predict_row(x, node.left)
        else:
            return self._predict_row(x, node.right)

    def predict(self, X):
        """Predict using the tree"""
        return np.array([self._predict_row(x, self.root) for x in X])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

class XGBoost:
    """XGBoost implementation (regression + optional binary classification)"""

    def __init__(
        self,
        n_estimators=50,
        learning_rate=0.1,
        max_depth=3,
        min_samples_leaf=20,
        lambda_=1.0,
        gamma=0.0,
        objective="reg:squarederror"  # or "binary:logistic"
    ):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.lambda_ = lambda_
        self.gamma = gamma
        self.objective = objective
        self.trees = []
        self.base_score = None  # initial prediction

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-x))

    def _compute_grad_hess(self, y, y_pred):
        if self.objective == "reg:squarederror":
            # Loss = 0.5 * (y - y_pred)^2
            # gradient dL/dy_pred = (y_pred - y), hessian = 1
            grad = y_pred - y
            hess = np.ones_like(y)
        elif self.objective == "binary:logistic":
            # Logistic loss with raw score y_pred
            p = self._sigmoid(y_pred)
            grad = p - y
            hess = p * (1.0 - p)
        else:
            raise ValueError("Unknown objective")
        return grad, hess

    def fit(self, X, y):
        """Train the XGBoost model"""
        n_samples = X.shape[0]

        # Initial prediction: mean for regression, log-odds for classification
        if self.objective == "reg:squarederror":
            self.base_score = np.mean(y)
        elif self.objective == "binary:logistic":
            pos_ratio = np.clip(y.mean(), 1e-6, 1 - 1e-6)
            self.base_score = np.log(pos_ratio / (1 - pos_ratio))

        y_pred = np.full(n_samples, self.base_score, dtype=float)
        self.trees = []

        for m in range(self.n_estimators):
            grad, hess = self._compute_grad_hess(y, y_pred)

            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                lambda_=self.lambda_,
                gamma=self.gamma,
            )
            tree.fit(X, grad, hess)
            update = tree.predict(X)

            y_pred += self.learning_rate * update
            self.trees.append(tree)

    def predict(self, X):
        """Make predictions (raw score for logistic, value for regression)"""
        pred = np.full(X.shape[0], self.base_score, dtype=float)
        for tree in self.trees:
            pred += self.learning_rate * tree.predict(X)

        if self.objective == "binary:logistic":
            # Return class labels
            proba = self._sigmoid(pred)
            return (proba >= 0.5).astype(int)
        else:
            return pred

    def predict_proba(self, X):
        """Predict probabilities for binary classification"""
        if self.objective != "binary:logistic":
            raise ValueError("predict_proba only valid for binary:logistic")
        pred = np.full(X.shape[0], self.base_score, dtype=float)
        for tree in self.trees:
            pred += self.learning_rate * tree.predict(X)
        proba = self._sigmoid(pred)
        # Return shape (n_samples, 2): P(class 0), P(class 1)
        return np.column_stack([1 - proba, proba])


In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
X = data.data
y = data.target

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train XGBoost-style regressor
model = XGBoost(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    min_samples_leaf=30,
    lambda_=1.0,
    gamma=0.0,
    objective="reg:squarederror",
)
model.fit(X_train, y_train)

# Evaluate on test set
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Test RMSE:", rmse)
print("Test R^2:", r2)


Test RMSE: 0.5797697132392867
Test R^2: 0.743490066100744


## XGBoost - Classification

* Training an XGBoost model on the `Breast Cancer` binary classification task. Reporting on the performance predicting unseen test samples.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X = data.data
y = data.target  # 0 malignant, 1 benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = XGBoost(
    n_estimators=60,
    learning_rate=0.1,
    max_depth=3,
    min_samples_leaf=10,
    lambda_=1.0,
    gamma=0.0,
    objective="binary:logistic",
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Test accuracy:", acc)

# probabilities and ROC-AUC
from sklearn.metrics import roc_auc_score
y_proba = clf.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, y_proba))


Test accuracy: 0.9649122807017544
ROC-AUC: 0.9943783068783069
